# الفصل 52: هندسة الأوامر المتقدمة

> قبل أي تقنية متقدمة (RAG، Agents)، يبقى الـ Prompt نفسه هو أهم "واجهة برمجة" بينك وبين LLM.

- **الجزء:** 10 — التطبيقات والوكلاء الذكية
- **الفصل:** 52
- **المستوى:** متقدم
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 51: الذاكرة طويلة المدى للوكلاء
- **الوسوم:** Chain-of-Thought، Self-Consistency، Tree-of-Thoughts، Prompting

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch52-advanced-prompting.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-10-applications-agents%2Fch52-advanced-prompting.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-10-applications-agents/ch52-advanced-prompting.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch52-advanced-prompting.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch52-advanced-prompting.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch52-advanced-prompting.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** pydantic
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("pydantic", "pydantic")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch52-advanced-prompting.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-10-applications-agents%2Fch52-advanced-prompting.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-10-applications-agents/ch52-advanced-prompting.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch52-advanced-prompting.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch52-advanced-prompting.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch52-advanced-prompting.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> قبل أي تقنية متقدمة (RAG، Agents)، يبقى الـ Prompt نفسه هو أهم "واجهة برمجة" بينك وبين LLM.

> **بيانات الفصل**
>
> - **الجزء:** 10 — التطبيقات والوكلاء الذكية
> - **المستوى:** متقدم
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 51: الذاكرة طويلة المدى للوكلاء


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم تقنيات الاستدلال المتسلسل (Chain-of-Thought وتطوراتها)
- فهم تقنيات هيكلة المخرجات (Structured Output)
- فهم Chain-of-Thought


## 1. لماذا يهمّك هذا الموضوع؟

قبل أي تقنية متقدمة (RAG، Agents)، يبقى الـ Prompt نفسه هو أهم "واجهة برمجة" بينك وبين LLM. إتقان هندسة الأوامر بعمق يوفّر عليك الكثير من التعقيد والتكلفة التي قد تحتاجها حلول أكثر تطورًا.


## 2. الخريطة الذهنية


```text
هندسة الأوامر
├── البنية الأساسية
│   ├── الدور والمهمة
│   ├── السياق والقيود
│   ├── شكل المخرجات
│   └── أمثلة توضيحية
├── الاستدلال
│   ├── Zero-shot CoT
│   ├── Few-shot CoT
│   ├── Self-Consistency
│   ├── Tree-of-Thoughts
│   └── ReAct (للوكلاء)
├── المخرجات المُهيكَلة
│   ├── JSON Schema
│   ├── Function Calling
│   └── التحقق وإعادة المحاولة
├── التحسين الآلي
│   ├── DSPy
│   ├── تحسين التعليمات
│   └── اختيار الأمثلة
└── التقييم
    ├── مجموعة اختبار ثابتة
    ├── حساسية الصياغة
    └── التكلفة والزمن
```


## 3. الشرح المتدرج

### 🟢 أ) تقنيات الاستدلال المتسلسل (Chain-of-Thought وتطوراتها)


In [ ]:
# 1) Zero-shot CoT — أبسط تقنية وأكثرها فعالية مقابل التكلفة
ZERO_SHOT_COT = "فكّر خطوة بخطوة."

# 2) Few-shot CoT — أمثلة توضح نمط الاستدلال المطلوب
FEW_SHOT_COT_EXAMPLE = """
سؤال: إذا كان لدى أحمد 5 تفاحات واشترى 3 أخرى ثم أكل تفاحتين، كم تفاحة لديه؟
الحل خطوة بخطوة:
1. البداية: 5 تفاحات
2. بعد الشراء: 5 + 3 = 8 تفاحات
3. بعد الأكل: 8 - 2 = 6 تفاحات
الإجابة: 6
"""

# 3) Self-Consistency — توليد عدة مسارات استدلال والتصويت بالأغلبية
def self_consistency_reasoning(llm, question: str, num_samples=5) -> str:
    """
    بدلاً من الاعتماد على مسار استدلال واحد (قد يحتوي خطأ عرضي)،
    نُولِّد عدة مسارات بـ temperature > 0 ونصوّت على الإجابة النهائية الأكثر تكرارًا
    """
    answers = []
    for _ in range(num_samples):
        response = llm.generate(f"{question}\n{ZERO_SHOT_COT}", temperature=0.7)
        final_answer = extract_final_answer(response)   # دالة استخراج مخصصة حسب التنسيق
        answers.append(final_answer)

    from collections import Counter
    most_common_answer, count = Counter(answers).most_common(1)[0]
    confidence = count / num_samples
    return {"answer": most_common_answer, "confidence": confidence, "all_answers": answers}

def extract_final_answer(response: str) -> str:
    if "الإجابة:" in response:
        return response.split("الإجابة:")[-1].strip().split("\n")[0]
    return response.strip()

# 4) Tree-of-Thoughts — استكشاف شجرة من مسارات الاستدلال مع تقييم وتراجع
class TreeOfThoughtsSolver:
    """
    بدلاً من مسار استدلال خطي واحد، نستكشف أشجارًا من "أفكار" جزئية
    مع تقييم كل فرع، والتراجع عن الأفرع الضعيفة (مشابه لـ BFS/DFS في البحث)
    """
    def __init__(self, llm, max_depth=3, branches_per_step=3):
        self.llm = llm
        self.max_depth = max_depth
        self.branches_per_step = branches_per_step

    def generate_next_thoughts(self, problem: str, current_path: list) -> list:
        prompt = f"""المسألة: {problem}
المسار الحالي من الأفكار: {current_path}

اقترح {self.branches_per_step} خطوات تفكير مختلفة ومحتملة للمتابعة من هنا."""
        response = self.llm.generate(prompt)
        return [line.strip() for line in response.split("\n") if line.strip()][:self.branches_per_step]

    def evaluate_path(self, problem: str, path: list) -> float:
        eval_prompt = f"""المسألة: {problem}
مسار التفكير: {path}

قيّم مدى واعدية هذا المسار للوصول لحل صحيح، من 0 إلى 1. أعطِ رقمًا فقط."""
        try:
            return float(self.llm.generate(eval_prompt).strip())
        except ValueError:
            return 0.5

    def solve(self, problem: str) -> dict:
        paths = [[]]   # نبدأ بمسار فارغ واحد

        for depth in range(self.max_depth):
            all_candidates = []
            for path in paths:
                next_thoughts = self.generate_next_thoughts(problem, path)
                for thought in next_thoughts:
                    new_path = path + [thought]
                    score = self.evaluate_path(problem, new_path)
                    all_candidates.append((new_path, score))

            # الاحتفاظ بأفضل المسارات فقط (Beam Search)
            all_candidates.sort(key=lambda x: x[1], reverse=True)
            paths = [path for path, score in all_candidates[:self.branches_per_step]]

        best_path = paths[0]
        final_prompt = f"المسألة: {problem}\nمسار التفكير النهائي: {best_path}\nاستنتج الإجابة النهائية:"
        return {"final_answer": self.llm.generate(final_prompt), "best_path": best_path}


### 🟡 ب) تقنيات هيكلة المخرجات (Structured Output)


In [ ]:
from pydantic import BaseModel, Field
from typing import List

class ExtractedInvoice(BaseModel):
    """تعريف صارم للمخرج المطلوب — يُستخدم مع Function Calling/JSON mode"""
    invoice_number: str = Field(description="رقم الفاتورة")
    total_amount: float = Field(description="المبلغ الإجمالي")
    items: List[str] = Field(description="قائمة العناصر المذكورة في الفاتورة")
    date: str = Field(description="تاريخ الفاتورة بصيغة YYYY-MM-DD")

def structured_extraction_with_validation(llm, document_text: str, schema: BaseModel) -> dict:
    """
    استخدام JSON Schema + التحقق (validation) لضمان مخرجات موثوقة
    قابلة للمعالجة البرمجية المباشرة دون تحليل نص هش
    """
    schema_json = schema.model_json_schema()
    prompt = f"""استخرج المعلومات التالية من النص وفق المخطط المحدَّد بالضبط:
المخطط: {schema_json}

النص: {document_text}

أعطِ النتيجة بصيغة JSON صحيحة فقط، بدون أي نص إضافي."""

    raw_output = llm.generate(prompt, response_format="json")
    try:
        validated = schema.model_validate_json(raw_output)
        return {"success": True, "data": validated.model_dump()}
    except Exception as e:
        # إعادة محاولة بتصحيح الخطأ (Self-correction loop)
        correction_prompt = f"""المخرج السابق لم يطابق المخطط المطلوب. الخطأ: {str(e)}
المخرج الخاطئ: {raw_output}
صحّح المخرج ليطابق المخطط بالضبط:"""
        corrected_output = llm.generate(correction_prompt, response_format="json")
        validated = schema.model_validate_json(corrected_output)
        return {"success": True, "data": validated.model_dump(), "required_correction": True}


### 🔴 ج) طبقة التعمّق: تحسين الأوامر آليًا (Automated Prompt Optimization)

بدلاً من هندسة الأوامر يدويًا بالتجربة والخطأ، أطر حديثة مثل **DSPy** تُعامل الـ prompt كـ "معامل قابل للتحسين" تلقائيًا:


In [ ]:
"""
توضيح مفاهيمي لفلسفة DSPy:
بدلاً من كتابة prompt ثابت، نُعرِّف "برنامجًا" من وحدات (modules) قابلة للتحسين،
ونسمح لإطار العمل بالبحث آليًا عن أفضل صياغة/أمثلة توضيحية بناءً على مقياس أداء محدَّد
"""

class OptimizablePromptModule:
    def __init__(self, llm, task_description: str, initial_instruction: str):
        self.llm = llm
        self.task_description = task_description
        self.instruction = initial_instruction
        self.few_shot_examples = []

    def predict(self, input_text: str) -> str:
        examples_text = "\n".join([
            f"مدخل: {ex['input']}\nمخرج: {ex['output']}" for ex in self.few_shot_examples
        ])
        prompt = f"{self.instruction}\n\n{examples_text}\n\nمدخل: {input_text}\nمخرج:"
        return self.llm.generate(prompt)

    def optimize(self, training_examples: list, metric_fn: callable, num_candidates=5):
        """
        بحث آلي مبسّط: نُجرِّب عدة صياغات تعليمات وأمثلة مختلفة،
        ونختار المجموعة التي تحقق أعلى درجة على مقياس الأداء المحدَّد
        """
        best_score, best_config = 0, None

        for _ in range(num_candidates):
            # توليد صياغة تعليمة بديلة واختيار عينة عشوائية من أمثلة توضيحية
            candidate_instruction = self.llm.generate(
                f"أعد صياغة هذه التعليمة بطريقة أوضح: {self.instruction}"
            )
            candidate_examples = random.sample(training_examples, min(3, len(training_examples)))

            self.instruction = candidate_instruction
            self.few_shot_examples = candidate_examples

            scores = [metric_fn(self.predict(ex["input"]), ex["output"]) for ex in training_examples[:20]]
            avg_score = sum(scores) / len(scores)

            if avg_score > best_score:
                best_score = avg_score
                best_config = {"instruction": candidate_instruction, "examples": candidate_examples}

        self.instruction = best_config["instruction"]
        self.few_shot_examples = best_config["examples"]
        return best_score


## 4. مثال محلول خطوة بخطوة

**المسألة:** «اشترى أحمد 3 كتب بسعر 45 ريالًا لكل كتاب، وحصل على خصم 10%، ودفع 150 ريالًا. كم الباقي؟»

**مسار CoT:**

1. تكلفة الكتب قبل الخصم: \(3 \times 45 = 135\) ريالًا.
2. قيمة الخصم: \(135 \times 0.10 = 13.5\) ريالًا.
3. التكلفة بعد الخصم: \(135 - 13.5 = 121.5\) ريالًا.
4. الباقي: \(150 - 121.5 = 28.5\) ريالًا.

**Self-Consistency:** نولّد 5 مسارات بـ `temperature=0.7`:

| المسار | الإجابة |
|---|---:|
| 1 | 28.5 |
| 2 | 28.5 |
| 3 | 27.0 (خطأ في الخصم) |
| 4 | 28.5 |
| 5 | 28.5 |

التصويت بالأغلبية يعطي 28.5 بثقة \(4/5 = 80\%\). لاحظ أن الإجابة الصحيحة لم تكن الوحيدة، لكن التصويت قلّل أثر الخطأ العرضي.

**التكلفة:** 5 استدعاءات بدلًا من 1. إذا كانت المسألة حرجة، فالثمن مقبول. إذا كانت المسألة بسيطة، فالـ CoT العادي كافٍ. القاعدة: **لا تدفع مقابل تنويع لا يغيّر القرار**.


## 5. نصائح وحيل

> 💡 **ضع التعليمات أولًا والسياق بعدها**: النماذج تعطي وزنًا أكبر لبداية الـ prompt. ابدأ بالمهمة والقيود، ثم ضع المستندات بين فواصل واضحة.

> 💡 **استخدم فواصل صريحة**: `### السياق` و`### المطلوب` و`### شكل المخرجات` تقلّل الخلط بين التعليمات والبيانات.

> 💡 **أعطِ مثالًا واحدًا جيدًا بدل خمسة متوسطة**: جودة المثال وتنوعه أهم من عدده. تأكد أنه لا يُسرّب الإجابة في مهام التقييم.

> 💡 **اطلب JSON وفق schema محددة**: وصف الحقول وأنواعها، ثم تحقق برمجيًا وأعد المحاولة عند الفشل. لا تعتمد على «أرجو أن يكون JSON صحيحًا».

> 💡 **اختبر حساسية الصياغة**: أعد صياغة الـ prompt بثلاث طرق طفيفة وقِس التباين في النتائج. الـ prompt الهشّ يفشل في الإنتاج.


## 6. أخطاء شائعة ومصائد

> ⚠️ **تعليمات متناقضة**: «كن موجزًا» و«اشرح كل خطوة بالتفصيل» في نفس الـ prompt. النموذج سيختار عشوائيًا. رتّب الأولويات صراحةً.

> ⚠️ **حقن الأوامر عبر المحتوى**: نص المستخدم أو المستند قد يحتوي «تجاهل ما سبق». ضع المحتوى داخل حدود، واذكر أن التعليمات داخله بيانات لا أوامر.

> ⚠️ **الإفراط في CoT مع نماذج الاستدلال**: بعض النماذج الحديثة تُفكّر داخليًا؛ إضافة «فكّر خطوة بخطوة» قد تُطيل المخرجات دون مكسب، أو تُضعف الأداء. اختبر قبل الاعتماد.

> ⚠️ **أمثلة Few-shot مسرّبة**: إذا كانت أمثلة التقييم موجودة في الـ prompt، فالنتيجة مضخّمة. افصل أمثلة التطوير عن مجموعة الاختبار.

> ⚠️ **عدم تثبيت الإصدار**: تغيير الـ prompt دون تسجيل يجعل تحليل الانحدار مستحيلًا. تعامل معه ككود: إصدار، مراجعة، واختبار.


## 7. جدول مقارنة

| التقنية | التكلفة (عدد استدعاءات LLM) | متى تستخدمها |
|---|---|---|
| **Zero-shot CoT** | 1 | خط أساسي سريع، يجب تجربته أولًا دائمًا |
| **Few-shot CoT** | 1 (لكن prompt أطول) | عندما يحتاج النموذج أمثلة لنمط استدلال محدّد |
| **Self-Consistency** | N (عادة 5-10) | مسائل حرجة تحتمل الخطأ العرضي، ولديك ميزانية حسابية |
| **Tree-of-Thoughts** | كبيرة جدًا (أُسّية تقريبًا) | مسائل تخطيط/ألغاز معقدة جدًا تحتاج استكشافًا فعليًا |
| **DSPy-style Auto-opt** | كبيرة لمرة واحدة (التحسين)، ثم 1 للاستخدام | تطبيقات إنتاجية تستحق الاستثمار في تحسين مُسبَق |


## 8. ملخص أفضل الممارسات

- [ ] ابدأ دائمًا بأبسط prompt ممكن، وأضف التعقيد فقط عند الحاجة المُثبَتة
- [ ] استخدم أمثلة توضيحية (few-shot) قريبة قدر الإمكان من التوزيع الفعلي للاستخدام
- [ ] لمخرجات تحتاج معالجة برمجية، استخدم دائمًا JSON mode/Function calling وليس تحليل نص حر
- [ ] اختبر حساسية النموذج لصياغة الـ prompt (robustness) عبر إعادة صياغات طفيفة
- [ ] وثّق كل نسخة من الـ prompt كأي كود آخر (version control)، فهو معامل حرج في النظام

---


## 9. أسئلة مقابلات

**سؤال 1: ما الفرق بين CoT وReAct؟**

> **إجابة نموذجية:** CoT يطلب من النموذج إظهار خطوات الاستدلال قبل الإجابة، لكنه يبقى داخل النموذج ولا يتفاعل مع العالم. ReAct يضيف دورة فعل-ملاحظة: النموذج يستدعي أداة، يرى نتيجتها، ثم يتابع. CoT مناسب للمسائل الاستدلالية، وReAct مناسب للمهام التي تحتاج معلومات خارجية أو تنفيذًا. يمكن دمجهما: تفكير يقرر الأداة، ثم ملاحظة تُغذّي التفكير.

**سؤال 2: متى تستخدم Self-Consistency؟**

> **إجابة نموذجية:** عندما تكون المسألة ذات إجابة نهائية قابلة للمقارنة (رقم، خيار، تصنيف)، ويكون الخطأ العرضي ممكنًا، وتتحمل ميزانية الحساب عدة استدعاءات. أستخدمها في التقييم أو القرارات المهمة، لا في كل طلب. أحسب عدد العينات بناءً على التباين: إذا كانت الإجابات متفقة في 3/3، فلا حاجة لـ 10.

**سؤال 3: كيف تضمن مخرجات JSON صحيحة؟**

> **إجابة نموذجية:** استخدم JSON mode أو Function Calling إن توفّر، وحدّد schema واضحة مع أنواع الحقول وقيم enum. تحقق برمجيًا باستخدام Pydantic أو JSON Schema، وأعد المحاولة مع رسالة خطأ محددة عند الفشل. لا تطلب «JSON فقط» دون تحقق؛ النموذج قد يضيف نصًا أو يحذف حقلًا. سجّل معدل الفشل لكل schema.


## 10. تمرين عملي

**الهدف:** مقارنة ثلاث استراتيجيات prompting على مهمة عربية واحدة.

**الخطوات:**
1. اختر 20 سؤالًا حسابيًا عربيًا قصيرًا بإجابات ذهبية.
2. نفّذ ثلاث نسخ: Zero-shot، Zero-shot CoT، وFew-shot CoT.
3. أضف نسخة رابعة: CoT + Self-Consistency (5 عينات).
4. احسب الدقة لكل نسخة، ومتوسط tokens، والزمن، والتكلفة.
5. أعد التجربة بصياغتين مختلفتين للسؤال لقياس الحساسية.
6. اكتب توصية: أي نسخة تختار لتطبيقك ولماذا.

**معيار النجاح:** تحسن CoT بدقة 10% على الأقل، ووضوح المقايضة بين الدقة والتكلفة في التقرير.


## 11. مراجع للتعمّق

- Wei, J., et al. (2022). *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models*. NeurIPS.
- Wang, X., et al. (2023). *Self-Consistency Improves Chain of Thought Reasoning in Language Models*. ICLR.
- Yao, S., et al. (2023). *Tree of Thoughts: Deliberate Problem Solving with Large Language Models*. NeurIPS.
- Khattab, O., et al. (2023). *DSPy: Compiling Declarative Language Model Calls into Self-Improving Pipelines*. ICLR 2024.
- Schulhoff, S., et al. (2024). *The Prompt Report: A Systematic Survey of Prompting Techniques*. arXiv:2406.06608.


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — التطبيقات والوكلاء الذكية / الفصل 52: هندسة الأوامر المتقدمة.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
